## Importing Libraries

In [4]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Loan Data').getOrCreate()

df = spark.read.option("header",True).csv("loan.csv")
df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Marital Status: string (nullable = true)
 |-- Family Size: string (nullable = true)
 |-- Income: string (nullable = true)
 |-- Expenditure: string (nullable = true)
 |-- Use Frequency: string (nullable = true)
 |-- Loan Category: string (nullable = true)
 |-- Loan Amount: string (nullable = true)
 |-- Overdue: string (nullable = true)
 |--  Debt Record: string (nullable = true)
 |--  Returned Cheque: string (nullable = true)
 |--  Dishonour of Bill: string (nullable = true)



## Data Cleaning

In [3]:
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import IntegerType

# Clean and convert relevant columns
df_cleaned = df.withColumn("Income", regexp_replace("Income", ",", "").cast(IntegerType())) \
    .withColumn("Expenditure", regexp_replace("Expenditure", ",", "").cast(IntegerType())) \
    .withColumn("Loan Amount", regexp_replace("Loan Amount", ",", "").cast(IntegerType())) \
    .withColumn("Debt Record", regexp_replace(" Debt Record", ",", "").cast(IntegerType())) \
    .withColumn("Returned Cheque", regexp_replace(" Returned Cheque", ",", "").cast(IntegerType())) \
    .withColumn("Overdue", regexp_replace("Overdue", ",", "").cast(IntegerType())) \
    .withColumn("Dishonour of Bill", regexp_replace(" Dishonour of Bill", ",", "").cast(IntegerType()))


### Number of Loans in Each Category

In [5]:
df_cleaned.groupBy("Loan Category").count().show()

+------------------+-----+
|     Loan Category|count|
+------------------+-----+
|           HOUSING|   67|
|        TRAVELLING|   53|
|       BOOK STORES|    7|
|       AGRICULTURE|   12|
|         GOLD LOAN|   77|
|  EDUCATIONAL LOAN|   20|
|        AUTOMOBILE|   60|
|          BUSINESS|   24|
|COMPUTER SOFTWARES|   35|
|           DINNING|   14|
|          SHOPPING|   35|
|       RESTAURANTS|   41|
|       ELECTRONICS|   14|
|          BUILDING|    7|
|        RESTAURANT|   20|
|   HOME APPLIANCES|   14|
+------------------+-----+



### Number of people who have taken more than 1 lack loan

In [6]:
df_cleaned.filter(col("Loan Amount") > 100000).count()

450

### Number of people with income greater than 60000 rupees

In [7]:
df_cleaned.filter(col("Income") > 60000).count()

198

### Number of people with 2 or more returned cheques and income less than 50000

In [8]:
df_cleaned.filter(
    (col("Returned Cheque") >= 2) & (col("Income") < 50000)
).count()

137

### Number of people with 2 or more returned cheques and are single

In [9]:
df_cleaned.filter(
    (col("Returned Cheque") >= 2) & (col("Marital Status") == "SINGLE")
).count()

111

### Number of people with expenditure over 50000 a month

In [10]:
df_cleaned.filter(col("Expenditure") > 50000).count()

6

### Number of members who are elgible for credit card

In [11]:
df_cleaned.filter(
    (col("Income") >= 45000) & 
    (col("Returned Cheque") <= 2) &
    (col("Overdue") <= 7)
).count()

101